# Local Research Companion — build notebook

Runs entirely on **Ollama** (no API tokens). This notebook is your workspace for Milestones 2–5.
Milestones 0 & 1 you've already verified with `milestone_0_1.py`, they're included here as quick re-checks.

**How to use this:** run the setup cell, then work down milestone by milestone. The plumbing cells
run as-is; the cells marked **TODO** are the actual lesson, fill them in. Keep `scraper.py` in the
same folder as this notebook.

| Milestone | What you implement |
|---|---|
| 0–1 | (done) smoke test + scrape |
| 2 | JSON link selection + a robust parser |
| 3 | streamed brochure |
| 4 | chat with memory |
| 5 | token budget trimming |

## Setup (run first)

In [1]:
import json
import requests
import tiktoken
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
import re

MODEL = "minimax-m3:cloud"          # "llama3.2:1b" if your machine is small
TOKEN_BUDGET = 3000

# Day 2 trick: OpenAI client pointed at local Ollama. No tokens spent.
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# tiktoken is OpenAI's tokenizer - an APPROXIMATION for llama, fine for budgeting.
_enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text):
    return len(_enc.encode(text))

print("Setup ready.")

Setup ready.


### Two LLM helpers we'll reuse

`chat()` for plain calls; `stream_to_notebook()` for the live typewriter effect (Day 5 style).

In [2]:
def chat(messages):
    """One non-streaming call. Returns the text."""
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

def stream_to_notebook(messages):
    """Streams the reply into a live-updating Markdown cell. Returns the full text."""
    stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
    result = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        update_display(Markdown(result), display_id=handle.display_id)
    return result

## Milestone 0 & 1 — quick re-check

In [3]:
# Smoke test
print(chat([{"role": "user", "content": "Tell me a fun fact in one sentence."}]))

Bananas are botanically classified as berries, while strawberries—despite their name—are not!


In [4]:
# Scrape check
TEST_URL = "https://anthropic.com"
links = fetch_website_links(TEST_URL)
print(f"{len(links)} links found. First few:")
for l in links[:8]:
    print("  -", l)

174 links found. First few:
  - #main
  - #footer
  - https://www.anthropic.com/
  - https://www.anthropic.com/research
  - https://www.anthropic.com/policy
  - https://www.anthropic.com/constitution
  - https://www.anthropic.com/claude-corps
  - https://www.anthropic.com/policy-on-the-ai-exponential


## Milestone 2 — relevant link selection (JSON)

The lesson: small local models are messier than frontier models at clean JSON. Implement
`safe_json()` to survive code fences and stray text, then `select_relevant_links()`.

First, **look at the raw output** before writing the parser, run the diagnostic cell below to see
exactly how `llama3.2` formats its JSON on your machine, then build the parser around what you see.

In [5]:
LINK_SYSTEM_PROMPT = """
TODO: Tell the model it receives a list of links and must respond with ONLY JSON of the
relevant ones for a company brochure (about, careers, products, company pages).
Required shape:
{"links": [{"type": "about page", "url": "https://full.url/about"}]}
Convert relative links to full https URLs. Exclude privacy/terms/email links.
"""

def links_user_prompt(url):
    found = fetch_website_links(url)
    return f"Links found on {url}:\n" + "\n".join(found)

In [6]:
# DIAGNOSTIC: see the raw model output before writing your parser.
raw = chat([
    {"role": "system", "content": LINK_SYSTEM_PROMPT},
    {"role": "user", "content": links_user_prompt(TEST_URL)},
])
print(repr(raw))   # repr() reveals fences/whitespace/newlines exactly

'```json\n{"links": [{"type": "about page", "url": "https://www.anthropic.com/company"}, {"type": "about page", "url": "https://www.anthropic.com/research"}, {"type": "careers page", "url": "https://www.anthropic.com/careers"}, {"type": "products page", "url": "https://claude.com/product/overview"}, {"type": "products page", "url": "https://claude.com/product/claude-code"}, {"type": "products page", "url": "https://claude.com/product/cowork"}, {"type": "products page", "url": "https://claude.com/product/claude-security"}, {"type": "products page", "url": "https://claude.com/platform/api"}, {"type": "products page", "url": "https://claude.com/pricing"}, {"type": "products page", "url": "https://www.anthropic.com/claude/opus"}, {"type": "products page", "url": "https://www.anthropic.com/claude/sonnet"}, {"type": "products page", "url": "https://www.anthropic.com/claude/haiku"}]}\n```'


In [7]:
def _first_json_object(text):
    """Return the first balanced {...} object, respecting quotes/escapes. None if absent."""
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        c = text[i]
        if escape:
            escape = False
            continue
        if c == "\\":
            escape = True
            continue
        if c == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None



def safe_json(text):
    if not text:
        return {"links": []}
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return {"links": []}
    try:
        data = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return {"links": []}
    return data if "links" in data else {"links": []}

def select_relevant_links(url):
    raw = chat([
        {"role": "system", "content": LINK_SYSTEM_PROMPT},
        {"role": "user", "content": links_user_prompt(url)},
    ])
    return safe_json(raw)

In [8]:
# Test it - should print a dict with a "links" list, and never crash
select_relevant_links(TEST_URL)

{'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/company'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/news'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/engineering'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/transparency'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/constitution'},
  {'type': 'careers', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'products', 'url': 'https://www.anthropic.com/claude/opus'},
  {'type': 'products', 'url': 'https://www.anthropic.com/claude/sonnet'},
  {'type': 'products', 'url': 'https://www.anthropic.com/claude/haiku'},
  {'type': 'products', 'url': 'https://www.anthropic.com/claude/mythos'},
  {'type': 'products', 'url': 'https://www.anthropic.com/claude/fable'},
  {'type': 'products', 'url': 'https://www.anthropic.com/product/claude-code'},
  {'type': 'products',
   'url': 'https://www.

In [9]:
# UPSTREAM DIAGNOSTIC — paste all of this output back

# 1. Are we even getting links to feed the model?
found = fetch_website_links(TEST_URL)
print("=== SCRAPED LINKS ===")
print("count:", len(found))
print(found[:15])

# 2. Is the system prompt actually asking for JSON?
print("\n=== SYSTEM PROMPT ===")
print(repr(LINK_SYSTEM_PROMPT))

# 3. What is the model returning RIGHT NOW?
raw = chat([
    {"role": "system", "content": LINK_SYSTEM_PROMPT},
    {"role": "user", "content": links_user_prompt(TEST_URL)},
])
print("\n=== RAW MODEL OUTPUT ===")
print(repr(raw))

# 4. What does the parser make of it?
print("\n=== PARSED ===")
print(safe_json(raw))

=== SCRAPED LINKS ===
count: 174
['#main', '#footer', 'https://www.anthropic.com/', 'https://www.anthropic.com/research', 'https://www.anthropic.com/policy', 'https://www.anthropic.com/constitution', 'https://www.anthropic.com/claude-corps', 'https://www.anthropic.com/policy-on-the-ai-exponential', 'https://www.anthropic.com/transparency', 'https://www.anthropic.com/responsible-scaling-policy', 'http://trust.anthropic.com/', 'https://www.anthropic.com/learn', 'https://claude.com/resources/tutorials', 'https://claude.com/resources/use-cases', 'https://www.anthropic.com/engineering']

=== SYSTEM PROMPT ===
'\nTODO: Tell the model it receives a list of links and must respond with ONLY JSON of the\nrelevant ones for a company brochure (about, careers, products, company pages).\nRequired shape:\n{"links": [{"type": "about page", "url": "https://full.url/about"}]}\nConvert relative links to full https URLs. Exclude privacy/terms/email links.\n'

=== RAW MODEL OUTPUT ===
'```json\n{\n  "links

## Milestone 3 — brochure, streamed

Assemble the landing page + relevant subpages into one context string (truncated so a small
model can cope), then stream the brochure.

In [10]:
BROCHURE_SYSTEM_PROMPT = """
TODO: Write a system prompt that turns the supplied page contents into a short markdown
brochure covering what the company does, its culture, customers, and careers. Markdown, no code blocks.
"""

def build_context(url):
    contents = fetch_website_contents(url)
    relevant = select_relevant_links(url)
    result = f"## Landing page:\n{contents}\n\n## Relevant pages:\n"
    for link in relevant["links"]:
        result += f"\n### {link['type']}\n" + fetch_website_contents(link["url"])
    return result[:4500]

In [11]:
def create_brochure(name, url):
    messages = [
        {"role": "system", "content": BROCHURE_SYSTEM_PROMPT},
        {"role": "user", "content": f"Company: {name}\n\n{build_context(url)}"},
    ]
    return stream_to_notebook(messages)

brochure = create_brochure("Anthropic", TEST_URL)

# Anthropic

## What We Do

Anthropic is an AI safety and research company dedicated to securing the benefits of artificial intelligence and mitigating its risks. As a public benefit corporation, we build AI systems that are **reliable, interpretable, and steerable** — designed to be helpful, honest, and harmless.

### Products & Models
- **Claude** — our flagship AI assistant
- **Claude Code**, **Claude Cowork**, **Claude Security**, and the **Claude Platform**
- A family of models including **Mythos, Fable, Opus, Sonnet, and Haiku**

### Approach to Safety
We treat AI safety as a systematic science. Our team conducts frontier research, applies a range of safety techniques, and feeds real-world insights from our products back into ongoing research. We openly share what we learn with the world.

## Our Culture

Anthropic is a collaborative, interdisciplinary team that brings together:
- Researchers
- Engineers
- Policy experts
- Business leaders and operators

We believe powerful AI requires more than technical excellence — it requires people who can connect research, engineering, policy, and operations. We view ourselves as one piece of a much larger puzzle and actively collaborate with **civil society, government, academia, nonprofits, and industry** to promote safety across the field.

Our work spans emerging areas including interpretability, reinforcement learning from human feedback (RLHF), and analysis of policy and societal impacts.

## Our Customers & Reach

Anthropic's work touches a global audience:
- Tens of thousands of individuals engaging with Claude daily
- Enterprise partners deploying Claude through the platform
- Public benefit initiatives like **Project Glasswing**, recently extended to roughly 150 new organizations across more than fifteen countries
- Researchers, policymakers, and institutions who rely on our published findings

We serve people, organizations, and society at large — guided by a commitment to ensuring powerful AI goes well for everyone.

## Careers

Anthropic is hiring researchers, engineers, and builders from a range of disciplines. If you're drawn to hard problems with real stakes, we'd like to meet you.

- **Mission:** Shape how AI meets the world
- **What you'll work on:** Frontier AI research, safety techniques, and products powered by Claude
- **Who we look for:** People with deep expertise and a sense of responsibility for how technology shapes society

Explore open roles at Anthropic and help build AI the world can rely on.

## Milestone 4 — chat with memory

Run the cell, then ask questions about the company. The model only "remembers" because we
resend the whole `messages` list each turn (Day 4). The TODO is the line that makes memory
persist, try commenting it out once to watch it forget.

In [12]:
def chat_loop(name, brochure):
    messages = [
        {"role": "system", "content": f"You are a helpful analyst answering questions about {name}."},
        {"role": "assistant", "content": brochure},
    ]
    print("Chat ready - type 'quit' to stop.")
    while True:
        question = input("you> ").strip()
        if question.lower() in {"quit", "exit"}:
            break
        messages.append({"role": "user", "content": question})
        messages = trim_history(messages)            # Milestone 5
        answer = stream_to_notebook(messages)
        # TODO (Milestone 4): append `answer` back as an assistant message so memory persists.
        messages.append({"role": "assistant", "content": answer})

## Milestone 5 — token budget

As the chat grows, every call costs more tokens. Implement `trim_history()` to keep the total
under `TOKEN_BUDGET` by dropping the oldest user/assistant pair, never the system message.
Define it **above** running the chat (cell order matters in notebooks).

In [13]:
def trim_history(messages):
    """
    TODO (Milestone 5): while total tokens > TOKEN_BUDGET, drop the oldest pair
    after the system message (i.e. messages[1] and messages[2]). Keep messages[0].
    Estimate with count_tokens() over the joined contents.
    """
    return messages   # replace me

# Once trim_history and the Milestone 4 TODO are done, start chatting:
chat_loop("Anthropic", brochure) 

Chat ready - type 'quit' to stop.


KeyboardInterrupt: Interrupted by user

## Stretch goals

Pick what interests you, each is a small, self-contained extension:
- **Two-model panel** — have `deepseek-r1:1.5b` critique the brochure, then `llama3.2` revise it.
- **Disk cache** — save scraped pages so re-runs are instant.
- **Compare two companies** in one run.
- **Gradio UI** — wraps this into a chat interface (and previews Week 2).
- **`/model` command** — switch models mid-chat.

When you've got the whole thing working, commit it: `git add . && git commit -m "Local Research Companion complete"`.